In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%sql
CREATE WIDGET TEXT storageName DEFAULT "adlsmartdataproy26";
CREATE WIDGET TEXT catalogo DEFAULT "catalog_ecobici_dev";


In [0]:
%python
dbutils.widgets.text("storageName", "adlsmartdataproy26")
dbutils.widgets.text("catalogo", "catalog_ecobici_dev")

storageName = dbutils.widgets.get("storageName")
catalogo    = dbutils.widgets.get("catalogo")

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credential)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-unit-catalog`
URL 'abfss://unit-catalog@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credential)
COMMENT 'Ubicación externa para las tablas unit-catalog del Data Lake';

In [0]:
%sql
-- ⚠️ SOLO para reiniciar el proyecto: borra todo el catálogo
-- DROP CATALOG IF EXISTS IDENTIFIER(:catalogo) CASCADE;

In [0]:
%python
spark.sql(f"""
CREATE CATALOG IF NOT EXISTS {catalogo}
MANAGED LOCATION 'abfss://unit-catalog@{storageName}.dfs.core.windows.net/{catalogo}'
""")

DataFrame[]

In [0]:
%python
info = spark.sql(f"DESCRIBE CATALOG EXTENDED {catalogo}").collect()

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalogo || '.raw');
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalogo || '.bronze');
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalogo || '.silver');
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalogo || '.golden');

In [0]:
%python
base = f"abfss://raw@{storageName}.dfs.core.windows.net/datasets"
 
for carpeta in ["viajes", "estaciones", "clima"]:
    dbutils.fs.mkdirs(f"{base}/{carpeta}")
 
display(dbutils.fs.ls(base))

path,name,size,modificationTime
abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/clima/,clima/,0,1794464522000
abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/estaciones/,estaciones/,0,1794464522000
abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/,viajes/,0,1794464522000
